# Tote-Sequencing Mixed-Integer Program

Minimise the **makespan** (time the last order finishes) by jointly deciding:
- **which slot** each tote occupies on the conveyor (`x[t,s]`)
- **which belt** each customer order is assigned to (`z[o,b]`)

Items are loaded onto the belt in item-index order within each tote.  
Every `δ = 3 s` one item passes; a belt is `τ = 6 s` downstream of the previous belt.  
The MIP is linearised with Big-M techniques following Option A from the formulation.

In [1]:
# ── Cell 1: Imports, parameters, and problem data ─────────────────────────────
import gurobipy as gp
from gurobipy import GRB
import pandas as pd

# Conveyor timing
delta = 3   # seconds to load one item onto the belt
tau   = 6   # transit seconds between adjacent belts

# Problem dimensions (1-indexed in the math; 0-indexed in Python)
n_items  = 7    # item types  i ∈ {1..7}
n_totes  = 11   # totes       t ∈ {1..11}
n_slots  = 11   # slots       s ∈ {1..11}  (bijection with totes)
n_belts  = 4    # belts       b ∈ {1..4}
n_orders = 9    # orders      o ∈ {1..9}

items  = range(n_items)
totes  = range(n_totes)
slots  = range(n_slots)
belts  = range(n_belts)
orders = range(n_orders)

# ── T[t][i] : quantity of item i in tote t ────────────────────────────────────
#    Rows = totes 1-11 | Columns = items 1-7
T = [
    [1, 0, 0, 0, 0, 0, 1],   # tote  1
    [0, 1, 0, 1, 0, 0, 0],   # tote  2
    [0, 0, 1, 0, 1, 0, 0],   # tote  3
    [1, 0, 0, 0, 0, 1, 0],   # tote  4
    [0, 1, 0, 0, 0, 0, 1],   # tote  5
    [0, 0, 0, 1, 0, 0, 0],   # tote  6
    [1, 0, 0, 0, 0, 0, 0],   # tote  7
    [0, 1, 0, 0, 0, 0, 0],   # tote  8
    [0, 0, 1, 0, 0, 0, 0],   # tote  9
    [0, 0, 0, 1, 0, 0, 1],   # tote 10
    [0, 0, 0, 0, 1, 1, 0],   # tote 11
]

# ── O[o][i] : quantity of item i required by order o ──────────────────────────
#    Rows = orders 1-9 | Columns = items 1-7
O = [
    [1, 0, 0, 1, 0, 0, 1],   # order 1  (items 1, 4, 7)
    [0, 1, 1, 0, 0, 0, 0],   # order 2  (items 2, 3)
    [0, 0, 0, 1, 1, 0, 0],   # order 3  (items 4, 5)
    [1, 1, 0, 0, 0, 0, 0],   # order 4  (items 1, 2)
    [0, 0, 1, 0, 0, 1, 0],   # order 5  (items 3, 6)
    [0, 0, 0, 0, 1, 1, 0],   # order 6  (items 5, 6)
    [1, 0, 0, 0, 0, 0, 1],   # order 7  (items 1, 7)
    [0, 1, 0, 1, 0, 0, 0],   # order 8  (items 2, 4)
    [0, 0, 0, 0, 0, 0, 1],   # order 9  (item  7)
]

# ── Verify conservation: total supply = total demand per item ─────────────────
supply = [sum(T[t][i] for t in totes) for i in items]
demand = [sum(O[o][i] for o in orders) for i in items]
assert supply == demand, f"Supply-demand mismatch!\n  supply={supply}\n  demand={demand}"

N_total = sum(supply)
print("Item |", " ".join(f"{i+1:>3}" for i in items))
print("Sup  |", " ".join(f"{supply[i]:>3}" for i in items))
print(f"\nTotal items in system  : {N_total}")
print(f"Totes / Slots          : {n_totes} / {n_slots}")
print(f"Belts / Orders         : {n_belts} / {n_orders}")


Item |   1   2   3   4   5   6   7
Sup  |   3   3   2   3   2   2   3

Total items in system  : 18
Totes / Slots          : 11 / 11
Belts / Orders         : 4 / 9


In [2]:
# ── Cell 2: Gurobi model and decision variables ───────────────────────────────

# Tight Big-M values
M_cnt  = N_total + 1                             # bound on any cumulative item count
M_time = (N_total - 1) * delta + (n_belts - 1) * tau  # bound on any arrival time

mdl = gp.Model("Tote_Sequencing_MIP")

# ── Core binary assignment variables ──────────────────────────────────────────
# x[t,s]  : 1 if tote t is placed in slot s
x = mdl.addVars(n_totes, n_slots, vtype=GRB.BINARY, name="x")

# z[o,b]  : 1 if order o is assigned to belt b
z = mdl.addVars(n_orders, n_belts, vtype=GRB.BINARY, name="z")

# ── Slot and belt aggregates ───────────────────────────────────────────────────
# S[s,i]  : quantity of item i in slot s  (derived from x)
S = mdl.addVars(n_slots, n_items, vtype=GRB.INTEGER, lb=0, name="S")

# B[b,i]  : total quantity of item i collected by belt b  (derived from z)
B = mdl.addVars(n_belts, n_items, vtype=GRB.INTEGER, lb=0, name="B")

# ── Sequence cumulative counts ─────────────────────────────────────────────────
# G[s,i]  : cumulative count of item i in slots 0..s
G = mdl.addVars(n_slots, n_items, vtype=GRB.INTEGER, lb=0, name="G")

# L[s]    : cumulative total items in slots 0..s
L = mdl.addVars(n_slots, vtype=GRB.INTEGER, lb=0, name="L")

# ── Completion-time computation chain ─────────────────────────────────────────
# m[b,i]  : global occurrence index of item i that belt b must collect last
#           = sum_{b'<=b} sum_o z[o,b'] * O[o,i]   (linear in z)
m = mdl.addVars(n_belts, n_items, vtype=GRB.INTEGER, lb=0, name="m")

# active[b,i] : 1 iff belt b needs at least one unit of item i  (m[b,i] >= 1)
#   needed so phi constraints are not imposed when m=0 (avoids infeasibility)
active = mdl.addVars(n_belts, n_items, vtype=GRB.BINARY, name="act")

# phi[b,i,s]  : 1 if the m[b,i]-th occurrence of item i falls in slot s
phi = mdl.addVars(n_belts, n_items, n_slots, vtype=GRB.BINARY, name="phi")

# Gamma[s,i]  : stream offset — items that arrive strictly before item i in slot s
#             = L[s-1] + sum_{i'<i} S[s,i'] - G[s-1,i]  >= 0
Gamma = mdl.addVars(n_slots, n_items, lb=0, name="Gam")

# mu[b,i,s]   : phi[b,i,s] * Gamma[s,i]   (binary x continuous linearisation)
mu = mdl.addVars(n_belts, n_items, n_slots, lb=0, name="mu")

# eta[b,i]    : stream position of the last unit of item i collected by belt b
#             = m[b,i] + Gamma[s*,i]  where s* is the selected slot
eta = mdl.addVars(n_belts, n_items, lb=0, name="eta")

# C[o,b]      : completion time of order o when assigned to belt b
C = mdl.addVars(n_orders, n_belts, lb=0, name="C")

# makespan    : objective variable
makespan = mdl.addVar(lb=0, name="MS")

mdl.update()
print(f"Variables  : {mdl.NumVars:,d}  "
      f"(binary={mdl.NumBinVars}, integer={mdl.NumIntVars}, "
      f"continuous={mdl.NumVars - mdl.NumBinVars - mdl.NumIntVars})")
print(f"Big-M (count) : {M_cnt}")
print(f"Big-M (time)  : {M_time} s")


Restricted license - for non-production use only - expires 2026-11-23
Variables  : 1,164  (binary=493, integer=714, continuous=-43)
Big-M (count) : 19
Big-M (time)  : 69 s


In [3]:
# ── Cell 3: Basic assignment and sequence constraints ─────────────────────────

# ── (1) Tote-slot bijection ────────────────────────────────────────────────────
# Each tote assigned to exactly one slot; each slot holds exactly one tote.
for t in totes:
    mdl.addConstr(gp.quicksum(x[t, s] for s in slots) == 1, name=f"tr_{t}")
for s in slots:
    mdl.addConstr(gp.quicksum(x[t, s] for t in totes) == 1, name=f"sc_{s}")

# ── (2) Slot item quantities: S[s,i] = sum_t T[t][i] * x[t,s] ────────────────
for s in slots:
    for i in items:
        mdl.addConstr(
            S[s, i] == gp.quicksum(T[t][i] * x[t, s] for t in totes),
            name=f"Sdef_{s}_{i}",
        )

# ── (3) Each order assigned to exactly one belt ───────────────────────────────
for o in orders:
    mdl.addConstr(gp.quicksum(z[o, b] for b in belts) == 1, name=f"ob_{o}")

# ── (4) Belt item totals: B[b,i] = sum_o O[o][i] * z[o,b] ───────────────────
for b in belts:
    for i in items:
        mdl.addConstr(
            B[b, i] == gp.quicksum(O[o][i] * z[o, b] for o in orders),
            name=f"Bdef_{b}_{i}",
        )

# ── (5) Conservation: sum_b B[b,i] = supply[i]  (redundant but tightens LP) ──
for i in items:
    mdl.addConstr(
        gp.quicksum(B[b, i] for b in belts) == supply[i],
        name=f"cons_{i}",
    )

# ── (6) Cumulative item counts G[s,i] and total-item counts L[s] ─────────────
for i in items:
    mdl.addConstr(G[0, i] == S[0, i], name=f"G0_{i}")
    for s in range(1, n_slots):
        mdl.addConstr(G[s, i] == G[s - 1, i] + S[s, i], name=f"G_{s}_{i}")

mdl.addConstr(L[0] == gp.quicksum(S[0, i] for i in items), name="L0")
for s in range(1, n_slots):
    mdl.addConstr(
        L[s] == L[s - 1] + gp.quicksum(S[s, i] for i in items),
        name=f"L_{s}",
    )

# ── (7) Stream offset: Gamma[s,i] = L[s-1] + sum_{i'<i} S[s,i'] - G[s-1,i] ──
# Interpretation: number of conveyor positions that pass belt before the
# *first* unit of item i in slot s appears in the stream.
for s in slots:
    L_prev = L[s - 1] if s > 0 else 0          # L_{s-1}; 0 when s=0
    for i in items:
        G_prev   = G[s - 1, i] if s > 0 else 0  # G_{s-1,i}; 0 when s=0
        before_i = (gp.quicksum(S[s, ip] for ip in range(i)) if i > 0 else 0)
        mdl.addConstr(
            Gamma[s, i] == L_prev + before_i - G_prev,
            name=f"Gam_{s}_{i}",
        )

mdl.update()
print(f"Constraints after basic block : {mdl.NumConstrs:,d}")


Constraints after basic block : 308


In [4]:
# ── Cell 4: Completion-time chain constraints (Option A Big-M linearisation) ──

# ── (8) m[b,i] = sum_{b'=0}^{b} sum_o z[o,b'] * O[o][i] ─────────────────────
# This is the index of the global occurrence of item i that belt b picks last.
# Belt b sees all units consumed by belts 0..b (R_{b,i} = demand from belts<b)
# plus its own demand, totalling the (R_{b,i}+O_{o,i})-th occurrence.
for b in belts:
    for i in items:
        mdl.addConstr(
            m[b, i] == gp.quicksum(
                O[o][i] * z[o, bp] for bp in range(b + 1) for o in orders
            ),
            name=f"m_{b}_{i}",
        )

# ── (9) active[b,i] = 1  iff  m[b,i] >= 1 ───────────────────────────────────
# Without this indicator the phi sum-to-1 constraint becomes infeasible when
# belt b has no order that needs item i (m=0 ⇒ no valid slot exists).
for b in belts:
    for i in items:
        # active=0  forces  m=0   (upper bound)
        mdl.addConstr(m[b, i] <= supply[i] * active[b, i], name=f"actU_{b}_{i}")
        # m=0  forces  active=0  (lower bound; m is integer ≥ 0)
        mdl.addConstr(active[b, i] <= m[b, i],              name=f"actL_{b}_{i}")

# ── (10) phi[b,i,s]: select the unique slot containing m[b,i]-th item-i ──────
# Exactly one slot selected when active=1; zero slots when active=0.
# Big-M constraints enforce:  G[s-1,i] < m[b,i] <= G[s,i]  when phi=1.
for b in belts:
    for i in items:
        mdl.addConstr(
            gp.quicksum(phi[b, i, s] for s in slots) == active[b, i],
            name=f"phiSum_{b}_{i}",
        )
        for s in slots:
            G_prev = G[s - 1, i] if s > 0 else 0   # G_{s-1,i}; constant 0 at s=0

            # G[s,i] >= m[b,i]      when phi=1  (m-th item reached by end of slot s)
            mdl.addConstr(
                G[s, i] >= m[b, i] - M_cnt * (1 - phi[b, i, s]),
                name=f"phiU_{b}_{i}_{s}",
            )
            # G[s-1,i] <= m[b,i]-1  when phi=1  (m-th item not yet reached before slot s)
            mdl.addConstr(
                G_prev <= m[b, i] - 1 + M_cnt * (1 - phi[b, i, s]),
                name=f"phiL_{b}_{i}_{s}",
            )

# ── (11) mu[b,i,s] = phi[b,i,s] * Gamma[s,i]  (binary × continuous) ─────────
# Standard McCormick envelope for binary × non-negative continuous variable.
#   phi=1 → mu = Gamma   |   phi=0 → mu = 0
for b in belts:
    for i in items:
        for s in slots:
            mdl.addConstr(mu[b, i, s] <= Gamma[s, i],
                          name=f"muA_{b}_{i}_{s}")
            mdl.addConstr(mu[b, i, s] <= M_cnt * phi[b, i, s],
                          name=f"muB_{b}_{i}_{s}")
            mdl.addConstr(mu[b, i, s] >= Gamma[s, i] - M_cnt * (1 - phi[b, i, s]),
                          name=f"muC_{b}_{i}_{s}")

# ── (12) eta[b,i] = m[b,i] + Gamma[s*, i]   (stream position) ───────────────
# Since exactly one phi is 1, the sum collapses to the single selected slot.
# eta[b,i] = n*_{m_{b,i}, i}  (global stream index of belt b's last unit of i)
for b in belts:
    for i in items:
        mdl.addConstr(
            eta[b, i] == m[b, i] + gp.quicksum(mu[b, i, s] for s in slots),
            name=f"eta_{b}_{i}",
        )

# ── (13) C[o,b]: completion time of order o when assigned to belt b ───────────
# Arrival time of stream position n at belt b: A(n,b) = (n-1)*delta + b*tau
# C[o,b] must be >= arrival of last needed unit of every item (when z[o,b]=1).
# Big-M relaxes the bound when z[o,b]=0 so inactive (o,b) pairs are harmless.
for o in orders:
    for b in belts:
        for i in items:
            if O[o][i] > 0:   # only items this order actually needs
                mdl.addConstr(
                    C[o, b] >= (eta[b, i] - 1) * delta + b * tau
                               - M_time * (1 - z[o, b]),
                    name=f"Clb_{o}_{b}_{i}",
                )

# ── (14) Makespan >= completion time of the assigned belt for every order ─────
for o in orders:
    for b in belts:
        mdl.addConstr(
            makespan >= C[o, b] - M_time * (1 - z[o, b]),
            name=f"MS_{o}_{b}",
        )

mdl.update()
print(f"Constraints after completion-time block : {mdl.NumConstrs:,d}")
print(f"Total variables : {mdl.NumVars:,d}  "
      f"(binary={mdl.NumBinVars}, integer={mdl.NumIntVars})")


Constraints after completion-time block : 2,096
Total variables : 1,164  (binary=493, integer=714)


In [5]:
# ── Cell 5: Objective and solve ───────────────────────────────────────────────

mdl.setObjective(makespan, GRB.MINIMIZE)

# Solver settings
mdl.Params.TimeLimit  = 300    # 5-minute wall-clock limit
mdl.Params.MIPGap     = 0.01   # stop at 1 % optimality gap
mdl.Params.OutputFlag = 1      # show Gurobi log

mdl.optimize()


Set parameter TimeLimit to value 300
Set parameter MIPGap to value 0.01
Set parameter OutputFlag to value 1
Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (mac64[x86] - Darwin 24.6.0 24G517)

CPU model: Intel(R) Core(TM) i7-1060NG7 CPU @ 1.20GHz
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  300
MIPGap  0.01



GurobiError: Model too large for size-limited license; visit https://gurobi.com/unrestricted for more information

In [ ]:
# ── Cell 6: Solution display ──────────────────────────────────────────────────

if mdl.Status in (GRB.OPTIMAL, GRB.TIME_LIMIT) and mdl.SolCount > 0:

    status_str = "OPTIMAL" if mdl.Status == GRB.OPTIMAL else "TIME LIMIT (best found)"
    print(f"{'='*62}")
    print(f"  Status   : {status_str}")
    print(f"  Makespan : {makespan.X:.1f} s")
    print(f"  MIP Gap  : {mdl.MIPGap * 100:.2f}%")
    print(f"{'='*62}\n")

    # ── Tote → Slot assignment ────────────────────────────────────────────────
    print("── Tote → Slot Assignment ──────────────────────────────────────")
    slot_rows = []
    for s in slots:
        for t in totes:
            if x[t, s].X > 0.5:
                contents = "  ".join(
                    f"i{i+1}×{T[t][i]}" for i in items if T[t][i] > 0
                )
                slot_rows.append({"Slot": s + 1, "Tote": t + 1, "Contents": contents})
    df_slots = pd.DataFrame(slot_rows).set_index("Slot")
    print(df_slots.to_string())

    # ── Order → Belt assignment + completion times ────────────────────────────
    print("\n── Order → Belt + Completion Times ────────────────────────────")
    order_rows = []
    for o in orders:
        for b in belts:
            if z[o, b].X > 0.5:
                needed = "  ".join(
                    f"i{i+1}×{O[o][i]}" for i in items if O[o][i] > 0
                )
                order_rows.append({
                    "Order": o + 1,
                    "Belt" : b + 1,
                    "Items": needed,
                    "C (s)": round(C[o, b].X, 1),
                })
    df_orders = pd.DataFrame(order_rows).set_index("Order")
    print(df_orders.to_string())

    # ── Item stream position diagnostics (per belt) ───────────────────────────
    print("\n── eta[b,i]: stream position of last collected unit ────────────")
    eta_rows = []
    for b in belts:
        row = {"Belt": b + 1}
        for i in items:
            row[f"i{i+1}"] = round(eta[b, i].X, 2) if active[b, i].X > 0.5 else "—"
        eta_rows.append(row)
    df_eta = pd.DataFrame(eta_rows).set_index("Belt")
    print(df_eta.to_string())

    print(f"\n{'='*62}")
    print(f"  Optimal Makespan : {makespan.X:.1f} s")
    print(f"{'='*62}")

else:
    print(f"No feasible solution found.  Gurobi status code: {mdl.Status}")
